# Multi-Crop Leaf Disease Detection - Google Colab Training (Storage-Optimized)

**This notebook trains MobileNetV2 and EfficientNet-Lite0 models using Colab's free temp storage + Google Drive.**

**Prerequisites:**
- Runtime -> Change runtime type -> GPU
- Dataset locally on your computer (processed/ folder with train/val/test splits)
- ~2-3 GB free space for temp dataset upload
- Google Drive mounted (for saving final models only)

**Key differences:**
- Dataset uploaded to Colab /content/ (temporary, auto-deleted)
- Models saved ONLY to Google Drive (~100 MB total)
- Downloads final models to your computer after training

## Step 1: Mount Google Drive (for saving models only)

In [ ]:
from google.colab import drive  # pyright: ignore[reportMissingImports]
drive.mount('/content/drive')
print("✓ Google Drive mounted (for saving final models only)")
print("Note: Dataset is uploaded directly to Colab, not stored on Drive")

## Step 2: Upload Dataset to Colab (temporary storage)

In [ ]:
import os
from google.colab import files  # pyright: ignore[reportMissingImports]


uploaded = files.upload()
if not uploaded:
    print("No file uploaded. Please try again.")
else:
    uploaded_files = list(uploaded.keys())
    print("Uploaded files:")
    for f in uploaded_files:
        print(f"  - {f}")

    # Helper: find 7z or split parts
    seven_parts = [f for f in uploaded_files if f.lower().endswith('.7z') or f.lower().endswith('.7z.001') or '.7z.' in f.lower()]
    zip_files = [f for f in uploaded_files if f.lower().endswith('.zip')]

    if seven_parts:
        print("\nDetected 7z archive or split parts. Installing p7zip and extracting...")
        os.system('apt-get update -qq && apt-get install -y p7zip-full > /dev/null')

        # Prefer .7z.001 as the entry point for split archives
        first = None
        for s in seven_parts:
            if s.lower().endswith('.7z.001'):
                first = s
                break
        if not first:
            # fallback to single .7z or the first available
            first = seven_parts[0]

        print(f"Extracting {first} to /content/processed/")
        os.makedirs('/content/processed', exist_ok=True)
        extract_cmd = f'7z x "{first}" -o/content/processed -y'
        ret = os.system(extract_cmd)
        if ret == 0:
            print("\n✓ Extraction complete: /content/processed/")
        else:
            print(f"\n✗ Extraction failed (exit code {ret}). You can try uploading all split parts and rerun this cell.")

    elif zip_files:
        zip_file = zip_files[0]
        print(f"\nDetected zip archive ({zip_file}). Extracting to /content/processed/")
        os.makedirs('/content/processed', exist_ok=True)
        os.system(f'unzip -q "{zip_file}" -d /content/processed')
        print("\n✓ Extraction complete: /content/processed/")

    else:
        # Try a generic fallback: if a .001 exists, attempt to extract it
        parts = [f for f in uploaded_files if f.endswith('.001')]
        if parts:
            first = parts[0]
            print("\nDetected split archive parts (.001 files). Installing p7zip and extracting...")
            os.system('apt-get update -qq && apt-get install -y p7zip-full > /dev/null')
            os.makedirs('/content/processed', exist_ok=True)
            ret = os.system(f'7z x "{first}" -o/content/processed -y')
            if ret == 0:
                print("\n✓ Extraction complete: /content/processed/")
            else:
                print(f"\n✗ Extraction failed (exit code {ret}).")
        else:
            print("\nUploaded files not recognized as .7z or .zip. Please upload a .7z (or split .7z.001/.7z.002...) or a .zip file.")

    # Verify structure
    dataset_base = "/content/processed"
    print("\nVerifying dataset structure:")
    for split in ['train', 'val', 'test']:
        split_path = os.path.join(dataset_base, split)
        if os.path.exists(split_path):
            n_classes = len([d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))])
            print(f"  ✓ {split}: {n_classes} classes")
        else:
            print(f"  ✗ {split}: NOT FOUND")


## Step 3: Clone Project Repository

In [ ]:
%cd /content

# Clone repo (replace with your GitHub repo URL)
!git clone https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git

%cd /content/multi-crop-leaf-disease-detection
print("Repository cloned successfully!")

## Step 4: Install Dependencies

In [ ]:
%pip install -q -r requirements.txt
print("Dependencies installed successfully!")

## Step 5: Configure Training for MobileNetV2

In [ ]:
import yaml
import os

# Paths - Dataset is now in Colab temp storage, models save to Drive
dataset_base = "/content/processed"  # Local Colab upload
output_base = "/content/drive/MyDrive/leaf_models"  # Save ONLY models here

# Load config
cfg_path = "training/config_mobilenetv2.yaml"
with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

# Set dataset paths (from local Colab upload)
cfg["dataset"]["data_dir"] = dataset_base
cfg["dataset"]["train_dir"] = f"{dataset_base}/train"
cfg["dataset"]["val_dir"] = f"{dataset_base}/val"
cfg["dataset"]["test_dir"] = f"{dataset_base}/test"

# Auto-count classes from training split
num_classes = len([
    d for d in os.listdir(cfg["dataset"]["train_dir"])
    if os.path.isdir(os.path.join(cfg["dataset"]["train_dir"], d))
])
cfg["model"]["num_classes"] = num_classes

# Set output paths - Models ONLY to Drive (skip logs to save space)
cfg["export"]["save_dir"] = f"{output_base}/mobilenetv2"

# Disable TensorBoard logging to save space
cfg["callbacks"]["tensorboard"]["enabled"] = False
cfg["callbacks"]["csv_logger"]["filename"] = f"{output_base}/training_log_mobilenetv2.csv"

# Save updated config
with open(cfg_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("✓ Configuration updated (storage-optimized)")
print(f"✓ Classes: {cfg['model']['num_classes']}")
print(f"✓ Train dataset: {cfg['dataset']['train_dir']}")
print(f"✓ Models save to: {cfg['export']['save_dir']}")
print(f"✓ TensorBoard logs: DISABLED (saves space)")

## Step 6: Train MobileNetV2

In [ ]:
%cd /content/multi-crop-leaf-disease-detection/training

!python train.py --config config_mobilenetv2.yaml

## Step 7: Configure & Train EfficientNet-Lite0

In [ ]:
import yaml
import os

# Load EfficientNet config
cfg_path = "config_efficientnet_lite0.yaml"
with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

# Paths
dataset_base = "/content/processed"  # Local Colab upload
output_base = "/content/drive/MyDrive/leaf_models"  # Save ONLY models

# Set dataset paths (from local Colab upload)
cfg["dataset"]["data_dir"] = dataset_base
cfg["dataset"]["train_dir"] = f"{dataset_base}/train"
cfg["dataset"]["val_dir"] = f"{dataset_base}/val"
cfg["dataset"]["test_dir"] = f"{dataset_base}/test"

# Auto-count classes
num_classes = len([
    d for d in os.listdir(cfg["dataset"]["train_dir"])
    if os.path.isdir(os.path.join(cfg["dataset"]["train_dir"], d))
])
cfg["model"]["num_classes"] = num_classes

# Set output paths
cfg["export"]["save_dir"] = f"{output_base}/efficientnet_lite0"

# Disable TensorBoard logging to save space
cfg["callbacks"]["tensorboard"]["enabled"] = False
cfg["callbacks"]["csv_logger"]["filename"] = f"{output_base}/training_log_efficientnet_lite0.csv"

# Save updated config
with open(cfg_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("✓ EfficientNet-Lite0 config updated")
print(f"✓ Classes: {cfg['model']['num_classes']}")
print(f"✓ Models save to: {cfg['export']['save_dir']}")

In [ ]:
!python train.py --config config_efficientnet_lite0.yaml

## Step 8: List Trained Models

In [ ]:
import os

models_dir = "/content/drive/MyDrive/leaf_models"

print("Trained models saved to Google Drive:\n")

for arch in ['mobilenetv2', 'efficientnet_lite0']:
    arch_dir = os.path.join(models_dir, arch)
    if os.path.exists(arch_dir):
        print(f"{arch}:")
        for f in os.listdir(arch_dir):
            fsize = os.path.getsize(os.path.join(arch_dir, f)) / (1024**2)
            print(f"  - {f} ({fsize:.1f} MB)")
    else:
        print(f"{arch}: No models found yet")

## Step 9: Evaluate a Trained Model

In [ ]:
import os
import subprocess

# Find the latest MobileNetV2 model
mobilenetv2_dir = "/content/drive/MyDrive/leaf_models/mobilenetv2"

if os.path.exists(mobilenetv2_dir):
    models = sorted([f for f in os.listdir(mobilenetv2_dir) if f.endswith('.h5')])
    if models:
        latest_model = models[-1]
        model_path = os.path.join(mobilenetv2_dir, latest_model)
        
        print(f"Evaluating: {latest_model}\n")
        
        training_dir = '/content/multi-crop-leaf-disease-detection/training'
        os.chdir(training_dir)
        subprocess.run(['python', 'evaluate.py', '--model', model_path, '--config', 'config_mobilenetv2.yaml'])
    else:
        print("No .h5 models found yet")
else:
    print(f"Directory not found: {mobilenetv2_dir}")

## Step 10: View Training Logs from Drive

In [ ]:
import pandas as pd
import os

results_dir = "/content/drive/MyDrive/leaf_models"

if os.path.exists(results_dir):
    csv_files = [f for f in os.listdir(results_dir) if f.endswith('.csv')]
    if csv_files:
        for csv_file in csv_files:
            csv_path = os.path.join(results_dir, csv_file)
            print(f"\n=== {csv_file} ===")
            df = pd.read_csv(csv_path)
            print(df.tail(10))  # Show last 10 rows
    else:
        print("No CSV files found yet")
else:
    print("Results directory not found yet")

## Step 11: Download Trained Models to Computer

In [ ]:
import os
import shutil
from google.colab import files  # pyright: ignore[reportMissingImports]

models_dir = "/content/drive/MyDrive/leaf_models"

print("=" * 60)
print("DOWNLOAD TRAINED MODELS")
print("=" * 60)

# Create a zip of all models
if os.path.exists(models_dir):
    print("\nPreparing models for download...")
    shutil.make_archive('leaf_models', 'zip', models_dir)
    
    # Show file sizes
    for arch in ['mobilenetv2', 'efficientnet_lite0']:
        arch_dir = os.path.join(models_dir, arch)
        if os.path.exists(arch_dir):
            for f in os.listdir(arch_dir):
                fpath = os.path.join(arch_dir, f)
                if os.path.isfile(fpath):
                    size = os.path.getsize(fpath) / (1024**2)
                    print(f"  {f}: {size:.1f} MB")
    
    print("\n✓ Downloading leaf_models.zip...")
    files.download('leaf_models.zip')
    print("✓ Download complete! Save this to your models/ folder")
else:
    print("No models found to download")

## Storage Summary

**What happened:**
- ✓ Dataset uploaded to Colab temp storage (automatically deleted after session)
- ✓ Models trained and saved to Google Drive (`/My Drive/leaf_models/`)
- ✓ Downloaded final models to your computer

**Google Drive usage:**
- Before: 7.16 GB
- After: ~7.2 GB (only added ~100 MB for final models)
- Savings: No dataset or logs stored on Drive!

**Next steps:**
1. Extract `leaf_models.zip` to your `models/` folder
2. Update your Flutter app to load models from local storage
3. Or, deploy models to your server for remote inference

**For future training:**
- Keep this notebook template
- Just re-upload dataset each session
- Old models stay on Drive as backups